# Future-Compatible Retrieval: Cross-Domain Base Experiment

This notebook builds the base caches and checkpoints used by the ETTh1/Weather mechanism diagnostics in this repository.

The central question is whether **future information available only during training** can supervise a relevance function that is evaluated using only observable past information at inference time.

For a query $q$ and historical candidate $i$, training-time future distance is

\[
d_f(q,i)=\frac{1}{H}\|\mathbf y_q-\mathbf y_i\|_2^2.
\]

The learned retriever approximates future-compatible relevance from observable query/candidate information only. The notebook evaluates ETTh1, Weather, Electricity, and Traffic at horizons $H\in\{24,48,96\}$ with past length $L=96$.

This base experiment is included because `01_etth1_weather_relevance.ipynb` reuses its windows, candidate pools, and checkpoints. The final paper's four-dataset confirmatory benchmark is reproduced separately in `experiments/confirmatory/confirmatory_benchmark.ipynb`.

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


## 0. Configuration

In [ ]:

from pathlib import Path
import copy
import gc
import math
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# ---------------------------------------------------------------------
# Exact paths: data are already downloaded.
# ---------------------------------------------------------------------

DATA_PATHS = {
    "ETTh1":
        REPO_DATA_ROOT / "ETT-small/ETTh1.csv",

    "Weather":
        REPO_DATA_ROOT / "weather/weather.csv",

    "Electricity":
        REPO_DATA_ROOT / "electricity/electricity.csv",

    "Traffic":
        REPO_DATA_ROOT / "traffic/traffic.csv",
}

RESULT_DIR = REPO_WORK_ROOT / "cross_domain_clean"

CACHE_DIR = (
    RESULT_DIR /
    "cache"
)

MODEL_DIR = (
    RESULT_DIR /
    "models"
)

for p in [
    RESULT_DIR,
    CACHE_DIR,
    MODEL_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

DATASET_NAMES = [
    "ETTh1",
    "Weather",
    "Electricity",
    "Traffic",
]

HORIZONS = [
    24,
    48,
    96,
]

SEQ_LEN = 96

TOP_M = 100
TOP_K = 10

TAU_Y = 0.50

SEEDS = [
    0,
    1,
    2,
]

# All channels for small datasets.
# Fixed deterministic subset for large datasets.
MAX_CHANNELS = {
    "ETTh1": None,
    "Weather": None,
    "Electricity": 32,
    "Traffic": 32,
}

# Reduce overlapping-window redundancy and runtime.
WINDOW_STRIDE = {
    "ETTh1": 8,
    "Weather": 24,
    "Electricity": 24,
    "Traffic": 24,
}

MAX_MEMORY_WINDOWS = 50000
MAX_TRAIN_QUERIES = 6000
MAX_VAL_QUERIES = 6000
MAX_TEST_QUERIES = 8000

TRAIN_BATCH = 128
EVAL_BATCH = 256
SEARCH_QUERY_BATCH = 512

MAX_EPOCHS = 30
PATIENCE = 6

LR = 1e-3
WEIGHT_DECAY = 1e-4

HAND_LAMBDA_GRID = [
    0.05,
    0.10,
    0.20,
    0.50,
    1.00,
]

BLOCK_ANCHORS = 10
N_BOOT = 3000

EPS = 1e-8

FORCE_REBUILD_WINDOWS = False
FORCE_REBUILD_PRESELECT = False
FORCE_RETRAIN = False

# Stable deterministic seeds.
DATASET_SEED = {
    "ETTh1": 1101,
    "Weather": 2202,
    "Electricity": 3303,
    "Traffic": 4404,
}

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
    print(
        "GPU memory (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory /
            1024**3,
            1,
        )
    )

print("Output:", RESULT_DIR)


## 1. Data validation

In [ ]:

for name, path in DATA_PATHS.items():

    assert path.exists(), (
        f"Missing dataset: {path}"
    )

    size_mb = (
        path.stat().st_size /
        1024**2
    )

    df_head = pd.read_csv(
        path,
        nrows=5,
    )

    print(
        f"{name:12s}",
        "|",
        f"{size_mb:8.2f} MB",
        "| first columns:",
        list(
            df_head.columns[
                :6
            ]
        ),
    )

print("\nAll four benchmark files found.")


## 2. Load numeric channels

In [ ]:

def load_numeric_csv(
    path,
):
    df = pd.read_csv(
        path
    )

    timestamp_cols = []

    for c in df.columns:

        if str(
            c
        ).lower() in {
            "date",
            "datetime",
            "timestamp",
            "time",
        }:
            timestamp_cols.append(
                c
            )

    x = (
        df
        .drop(
            columns=timestamp_cols,
            errors="ignore",
        )
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    # Keep columns that are essentially numeric.
    good_cols = [
        c
        for c in x.columns
        if x[
            c
        ].notna().mean() >
        0.99
    ]

    x = x[
        good_cols
    ]

    assert x.shape[
        1
    ] > 0

    x = (
        x
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .interpolate(
            axis=0,
            limit_direction="both",
        )
        .ffill()
        .bfill()
    )

    arr = x.to_numpy(
        dtype=np.float32
    )

    assert np.isfinite(
        arr
    ).all()

    return x


RAW = {}

for name in DATASET_NAMES:

    x = load_numeric_csv(
        DATA_PATHS[
            name
        ]
    )

    RAW[
        name
    ] = x

    print(
        f"{name:12s}",
        "| shape:",
        x.shape
    )


## 3. Chronological splits and channel selection

In [ ]:

def split_boundaries(
    dataset_name,
    n,
):
    if dataset_name == "ETTh1":

        # Standard ETT hourly split:
        # 12 months train / 4 months val / 4 months test.
        train_end = (
            12 *
            30 *
            24
        )

        val_end = (
            train_end +
            4 *
            30 *
            24
        )

        assert val_end < n

    else:

        # Standard generic split used by many LTSF benchmarks.
        train_end = int(
            0.70 *
            n
        )

        val_end = int(
            0.80 *
            n
        )

    return {
        "train_end":
            train_end,

        "val_end":
            val_end,

        "test_end":
            n,

        # Inner training retrieval memory/query boundary.
        "inner_memory_end":
            int(
                0.60 *
                train_end
            ),
    }


def select_channels(
    dataset_name,
    columns,
):
    cols = list(
        columns
    )

    max_c = MAX_CHANNELS[
        dataset_name
    ]

    if (
        max_c is None
        or
        len(
            cols
        ) <=
        max_c
    ):
        return cols

    idx = np.linspace(
        0,
        len(
            cols
        ) - 1,
        max_c,
        dtype=int,
    )

    return [
        cols[
            i
        ]
        for i in idx
    ]


SPLITS = {}
SELECTED_CHANNELS = {}

rows = []

for name in DATASET_NAMES:

    SPLITS[
        name
    ] = split_boundaries(
        name,
        len(
            RAW[
                name
            ]
        ),
    )

    SELECTED_CHANNELS[
        name
    ] = select_channels(
        name,
        RAW[
            name
        ].columns,
    )

    rows.append({
        "Dataset":
            name,

        "Rows":
            len(
                RAW[
                    name
                ]
            ),

        "OriginalChannels":
            RAW[
                name
            ].shape[
                1
            ],

        "UsedChannels":
            len(
                SELECTED_CHANNELS[
                    name
                ]
            ),

        **SPLITS[
            name
        ],
    })

split_table = pd.DataFrame(
    rows
)

display(
    split_table
)

split_table.to_csv(
    RESULT_DIR /
    "01_dataset_split_summary.csv",
    index=False,
)


## 4. Train-only channel normalization

In [ ]:

CHANNEL_NORMALIZED = {}

for name in DATASET_NAMES:

    cols = SELECTED_CHANNELS[
        name
    ]

    df = RAW[
        name
    ][
        cols
    ]

    train_end = SPLITS[
        name
    ][
        "train_end"
    ]

    train = df.iloc[
        :train_end
    ]

    mean = train.mean(
        axis=0
    ).to_numpy(
        dtype=np.float32
    )

    std = train.std(
        axis=0,
        ddof=0,
    ).to_numpy(
        dtype=np.float32
    )

    std = np.where(
        std <
        1e-6,
        1.0,
        std,
    ).astype(
        np.float32
    )

    arr = df.to_numpy(
        dtype=np.float32
    )

    z = (
        arr -
        mean[
            None,
            :
        ]
    ) / std[
        None,
        :
    ]

    assert np.isfinite(
        z
    ).all()

    CHANNEL_NORMALIZED[
        name
    ] = z.astype(
        np.float32
    )


## 5. Generic observable-past context

In [ ]:

def safe_autocorr_lag1(
    x,
):
    a = (
        x[
            :-1
        ] -
        x[
            :-1
        ].mean()
    )

    b = (
        x[
            1:
        ] -
        x[
            1:
        ].mean()
    )

    den = (
        np.sqrt(
            np.sum(
                a ** 2
            ) *
            np.sum(
                b ** 2
            )
        ) +
        EPS
    )

    return float(
        np.sum(
            a *
            b
        ) /
        den
    )


def normalized_slope(
    x,
):
    n = len(
        x
    )

    t = np.linspace(
        -1.0,
        1.0,
        n,
        dtype=np.float32,
    )

    t = (
        t -
        t.mean()
    )

    xc = (
        x -
        x.mean()
    )

    slope = (
        np.sum(
            t *
            xc
        ) /
        (
            np.sum(
                t ** 2
            ) +
            EPS
        )
    )

    return float(
        slope /
        (
            x.std() +
            EPS
        )
    )


def generic_context(
    past,
):
    past = np.asarray(
        past,
        dtype=np.float32,
    )

    L = len(
        past
    )

    short = max(
        8,
        L //
        4
    )

    std_full = (
        float(
            np.std(
                past
            )
        ) +
        EPS
    )

    current_level = (
        past[
            -1
        ] -
        np.mean(
            past
        )
    ) / std_full

    mean_gap = (
        np.mean(
            past[
                -short:
            ]
        ) -
        np.mean(
            past
        )
    ) / std_full

    short_change = (
        past[
            -1
        ] -
        past[
            -short
        ]
    ) / std_full

    long_change = (
        past[
            -1
        ] -
        past[
            0
        ]
    ) / std_full

    d_full = np.diff(
        past
    )

    d_short = np.diff(
        past[
            -short:
        ]
    )

    vol_ratio = (
        np.std(
            d_short
        ) +
        EPS
    ) / (
        np.std(
            d_full
        ) +
        EPS
    )

    slope = normalized_slope(
        past
    )

    ac1 = safe_autocorr_lag1(
        past
    )

    return np.asarray(
        [
            current_level,
            mean_gap,
            short_change,
            long_change,
            vol_ratio,
            slope,
            ac1,
        ],
        dtype=np.float32,
    )


CONTEXT_DIM = 7


## 6. Window construction

In [ ]:

def pattern_vector(
    past,
):
    x = np.asarray(
        past,
        dtype=np.float32,
    )

    x = (
        x -
        x.mean()
    )

    norm = np.linalg.norm(
        x
    )

    if norm < EPS:

        return np.zeros_like(
            x
        )

    return (
        x /
        norm
    ).astype(
        np.float32
    )


def build_windows(
    dataset_name,
    H,
):
    arr = CHANNEL_NORMALIZED[
        dataset_name
    ]

    channels = SELECTED_CHANNELS[
        dataset_name
    ]

    stride = WINDOW_STRIDE[
        dataset_name
    ]

    n_time, n_chan = arr.shape

    meta_rows = []
    pattern_rows = []
    context_rows = []
    future_rows = []

    for cidx in range(
        n_chan
    ):

        series = arr[
            :,
            cidx
        ]

        for anchor in range(
            SEQ_LEN,
            n_time -
            H +
            1,
            stride,
        ):

            past = series[
                anchor -
                SEQ_LEN:
                anchor
            ]

            future_raw = series[
                anchor:
                anchor +
                H
            ]

            past_std = float(
                np.std(
                    past
                )
            )

            if past_std < 1e-5:
                continue

            # Future path is expressed relative to last observation
            # and scaled only using observable past statistics.
            future = (
                future_raw -
                past[
                    -1
                ]
            ) / (
                past_std +
                EPS
            )

            meta_rows.append(
                (
                    cidx,
                    channels[
                        cidx
                    ],
                    anchor,
                    anchor +
                    H -
                    1,
                )
            )

            pattern_rows.append(
                pattern_vector(
                    past
                )
            )

            context_rows.append(
                generic_context(
                    past
                )
            )

            future_rows.append(
                future.astype(
                    np.float32
                )
            )

    return {
        "meta":
            pd.DataFrame(
                meta_rows,
                columns=[
                    "ChannelIndex",
                    "Channel",
                    "Anchor",
                    "FutureEnd",
                ],
            ),

        "pattern":
            np.stack(
                pattern_rows
            ).astype(
                np.float32
            ),

        "context":
            np.stack(
                context_rows
            ).astype(
                np.float32
            ),

        "future":
            np.stack(
                future_rows
            ).astype(
                np.float32
            ),
    }


## 7. Build/load window caches

In [ ]:

WINDOWS = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        npz_path = (
            CACHE_DIR /
            f"{dataset_name}_H{H}_windows.npz"
        )

        meta_path = (
            CACHE_DIR /
            f"{dataset_name}_H{H}_meta.parquet"
        )

        if (
            npz_path.exists()
            and
            meta_path.exists()
            and
            not
            FORCE_REBUILD_WINDOWS
        ):

            z = np.load(
                npz_path
            )

            w = {
                "meta":
                    pd.read_parquet(
                        meta_path
                    ),

                "pattern":
                    z[
                        "pattern"
                    ].astype(
                        np.float32
                    ),

                "context":
                    z[
                        "context"
                    ].astype(
                        np.float32
                    ),

                "future":
                    z[
                        "future"
                    ].astype(
                        np.float32
                    ),
            }

            print(
                "Loaded",
                dataset_name,
                "H=",
                H,
                "|",
                len(
                    w[
                        "meta"
                    ]
                ),
            )

        else:

            print(
                "Building",
                dataset_name,
                "H=",
                H
            )

            w = build_windows(
                dataset_name,
                H,
            )

            np.savez_compressed(
                npz_path,
                pattern=w[
                    "pattern"
                ],
                context=w[
                    "context"
                ],
                future=w[
                    "future"
                ],
            )

            w[
                "meta"
            ].to_parquet(
                meta_path,
                index=False,
            )

        WINDOWS[
            key
        ] = w


## 8. Strict temporal retrieval phases

In [ ]:

def build_phase_indices(
    dataset_name,
    H,
):
    w = WINDOWS[
        (
            dataset_name,
            H
        )
    ]

    meta = w[
        "meta"
    ]

    s = SPLITS[
        dataset_name
    ]

    anchor = meta[
        "Anchor"
    ].to_numpy()

    future_end = meta[
        "FutureEnd"
    ].to_numpy()

    train_memory = np.where(
        future_end <
        s[
            "inner_memory_end"
        ]
    )[0]

    train_query = np.where(
        (
            anchor >=
            s[
                "inner_memory_end"
            ]
        )
        &
        (
            future_end <
            s[
                "train_end"
            ]
        )
    )[0]

    val_memory = np.where(
        future_end <
        s[
            "train_end"
        ]
    )[0]

    val_query = np.where(
        (
            anchor >=
            s[
                "train_end"
            ]
        )
        &
        (
            future_end <
            s[
                "val_end"
            ]
        )
    )[0]

    test_memory = np.where(
        future_end <
        s[
            "val_end"
        ]
    )[0]

    test_query = np.where(
        anchor >=
        s[
            "val_end"
        ]
    )[0]

    return {
        "train_memory":
            train_memory,

        "train_query":
            train_query,

        "val_memory":
            val_memory,

        "val_query":
            val_query,

        "test_memory":
            test_memory,

        "test_query":
            test_query,
    }


PHASES = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        PHASES[
            (
                dataset_name,
                H
            )
        ] = build_phase_indices(
            dataset_name,
            H,
        )


## 9. Stable deterministic subsampling

In [ ]:

def deterministic_subset(
    indices,
    max_n,
    seed,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    if len(
        indices
    ) <= max_n:

        return np.sort(
            indices
        )

    rng = np.random.default_rng(
        seed
    )

    return np.sort(
        rng.choice(
            indices,
            size=max_n,
            replace=False,
        )
    )


TASK_INDICES = {}
sample_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        p = PHASES[
            key
        ]

        base = (
            DATASET_SEED[
                dataset_name
            ] +
            H *
            10
        )

        sampled = {
            "train_memory":
                deterministic_subset(
                    p[
                        "train_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    1,
                ),

            "train_query":
                deterministic_subset(
                    p[
                        "train_query"
                    ],
                    MAX_TRAIN_QUERIES,
                    base +
                    2,
                ),

            "val_memory":
                deterministic_subset(
                    p[
                        "val_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    3,
                ),

            "val_query":
                deterministic_subset(
                    p[
                        "val_query"
                    ],
                    MAX_VAL_QUERIES,
                    base +
                    4,
                ),

            "test_memory":
                deterministic_subset(
                    p[
                        "test_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    5,
                ),

            "test_query":
                deterministic_subset(
                    p[
                        "test_query"
                    ],
                    MAX_TEST_QUERIES,
                    base +
                    6,
                ),
        }

        TASK_INDICES[
            key
        ] = sampled

        sample_rows.append({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            **{
                k:
                    len(
                        v
                    )
                for k, v
                in sampled.items()
            },
        })

sample_table = pd.DataFrame(
    sample_rows
)

display(
    sample_table
)

sample_table.to_csv(
    RESULT_DIR /
    "02_task_sample_summary.csv",
    index=False,
)


## 10. Train-memory robust context scaler

In [ ]:

def fit_robust_scaler(
    x,
):
    med = np.median(
        x,
        axis=0,
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75 -
        q25
    )

    iqr = np.where(
        iqr <
        1e-5,
        1.0,
        iqr,
    )

    return (
        med.astype(
            np.float32
        ),
        iqr.astype(
            np.float32
        ),
    )


def apply_robust_scaler(
    x,
    med,
    iqr,
):
    z = (
        x -
        med
    ) / iqr

    z = np.clip(
        z,
        -8.0,
        8.0,
    )

    assert np.isfinite(
        z
    ).all()

    return z.astype(
        np.float32
    )


CONTEXT_SCALED = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        w = WINDOWS[
            key
        ]

        train_mem = TASK_INDICES[
            key
        ][
            "train_memory"
        ]

        med, iqr = fit_robust_scaler(
            w[
                "context"
            ][
                train_mem
            ]
        )

        CONTEXT_SCALED[
            key
        ] = apply_robust_scaler(
            w[
                "context"
            ],
            med,
            iqr,
        )


## 11. GPU Pattern Top-M search

In [ ]:

@torch.no_grad()
def pattern_topm_search(
    candidate_pattern,
    query_pattern,
    top_m,
):
    cand = torch.tensor(
        candidate_pattern,
        dtype=torch.float32,
        device=DEVICE,
    )

    query = torch.tensor(
        query_pattern,
        dtype=torch.float32,
        device=DEVICE,
    )

    idx_out = []
    score_out = []

    for start in range(
        0,
        len(
            query_pattern
        ),
        SEARCH_QUERY_BATCH,
    ):

        end = min(
            start +
            SEARCH_QUERY_BATCH,
            len(
                query_pattern
            ),
        )

        sim = (
            query[
                start:end
            ]
            @
            cand.T
        )

        score, idx = torch.topk(
            sim,
            k=min(
                top_m,
                cand.shape[
                    0
                ]
            ),
            dim=1,
            largest=True,
        )

        idx_out.append(
            idx.cpu()
        )

        score_out.append(
            score.cpu()
        )

        del sim

    return (
        torch.cat(
            idx_out,
            dim=0,
        ).numpy().astype(
            np.int64
        ),

        torch.cat(
            score_out,
            dim=0,
        ).numpy().astype(
            np.float32
        ),
    )


## 12. Build/load Pattern Top-M pools

In [ ]:

PRESELECT = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        w = WINDOWS[
            key
        ]

        idx = TASK_INDICES[
            key
        ]

        cache_path = (
            CACHE_DIR /
            f"{dataset_name}_H{H}_topM.npz"
        )

        if (
            cache_path.exists()
            and
            not
            FORCE_REBUILD_PRESELECT
        ):

            z = np.load(
                cache_path
            )

            out = {
                k:
                    z[
                        k
                    ]
                for k
                in z.files
            }

            print(
                "Loaded Top-M:",
                dataset_name,
                H
            )

        else:

            out = {}

            for phase in [
                "train",
                "val",
                "test",
            ]:

                mem_idx = idx[
                    f"{phase}_memory"
                ]

                q_idx = idx[
                    f"{phase}_query"
                ]

                local_idx, score = pattern_topm_search(
                    w[
                        "pattern"
                    ][
                        mem_idx
                    ],
                    w[
                        "pattern"
                    ][
                        q_idx
                    ],
                    TOP_M,
                )

                out[
                    f"{phase}_idx"
                ] = mem_idx[
                    local_idx
                ]

                out[
                    f"{phase}_score"
                ] = score

                out[
                    f"{phase}_query"
                ] = q_idx

            np.savez_compressed(
                cache_path,
                **out,
            )

            print(
                "Built Top-M:",
                dataset_name,
                H
            )

        PRESELECT[
            key
        ] = out


## 13. Phase tensor packaging

In [ ]:

def phase_data(
    dataset_name,
    H,
    phase,
):
    key = (
        dataset_name,
        H
    )

    w = WINDOWS[
        key
    ]

    p = PRESELECT[
        key
    ]

    q_idx = p[
        f"{phase}_query"
    ]

    cand_idx = p[
        f"{phase}_idx"
    ]

    meta = w[
        "meta"
    ]

    return {
        "q_idx":
            q_idx,

        "cand_idx":
            cand_idx,

        "pattern_score":
            p[
                f"{phase}_score"
            ],

        "q_context":
            CONTEXT_SCALED[
                key
            ][
                q_idx
            ],

        "cand_context":
            CONTEXT_SCALED[
                key
            ][
                cand_idx
            ],

        "q_future":
            w[
                "future"
            ][
                q_idx
            ],

        "cand_future":
            w[
                "future"
            ][
                cand_idx
            ],

        "q_anchor":
            meta.iloc[
                q_idx
            ][
                "Anchor"
            ].to_numpy(
                dtype=np.int64
            ),

        "q_channel":
            meta.iloc[
                q_idx
            ][
                "ChannelIndex"
            ].to_numpy(
                dtype=np.int64
            ),

        "cand_channel":
            meta.iloc[
                cand_idx.reshape(
                    -1
                )
            ][
                "ChannelIndex"
            ].to_numpy(
                dtype=np.int64
            ).reshape(
                cand_idx.shape
            ),
    }


## 14. Retrieval metrics

In [ ]:

def future_distance(
    q_future,
    cand_future,
):
    return np.mean(
        (
            cand_future -
            q_future[
                :,
                None,
                :
            ]
        ) ** 2,
        axis=2,
    )


def topk_from_scores(
    score,
    k,
):
    idx = np.argpartition(
        -score,
        kth=k -
        1,
        axis=1,
    )[
        :,
        :k
    ]

    row = np.arange(
        len(
            score
        )
    )[
        :,
        None
    ]

    local_score = score[
        row,
        idx
    ]

    order = np.argsort(
        -local_score,
        axis=1,
    )

    return idx[
        row,
        order
    ]


def gather2(
    x,
    idx,
):
    row = np.arange(
        len(
            x
        )
    )[
        :,
        None
    ]

    return x[
        row,
        idx
    ]


def gather3(
    x,
    idx,
):
    row = np.arange(
        len(
            x
        )
    )[
        :,
        None
    ]

    return x[
        row,
        idx,
        :
    ]


def ndcg_at_k(
    score,
    fdist,
    k,
):
    mean = fdist.mean(
        axis=1,
        keepdims=True,
    )

    std = (
        fdist.std(
            axis=1,
            keepdims=True,
        ) +
        1e-6
    )

    z = (
        fdist -
        mean
    ) / std

    relevance = np.exp(
        -z /
        TAU_Y
    )

    selected = topk_from_scores(
        score,
        k,
    )

    ideal = np.argsort(
        -relevance,
        axis=1,
    )[
        :,
        :k
    ]

    rel_sel = gather2(
        relevance,
        selected,
    )

    rel_ideal = gather2(
        relevance,
        ideal,
    )

    discount = (
        1.0 /
        np.log2(
            np.arange(
                2,
                k +
                2
            )
        )
    )[
        None,
        :
    ]

    dcg = np.sum(
        rel_sel *
        discount,
        axis=1,
    )

    idcg = (
        np.sum(
            rel_ideal *
            discount,
            axis=1,
        ) +
        EPS
    )

    return (
        dcg /
        idcg
    ).astype(
        np.float32
    )


def oracle_recall_at_k(
    selected,
    fdist,
    k,
):
    oracle = np.argpartition(
        fdist,
        kth=k -
        1,
        axis=1,
    )[
        :,
        :k
    ]

    out = np.zeros(
        len(
            selected
        ),
        dtype=np.float32,
    )

    for i in range(
        len(
            selected
        )
    ):

        out[
            i
        ] = (
            len(
                set(
                    selected[
                        i
                    ].tolist()
                )
                &
                set(
                    oracle[
                        i
                    ].tolist()
                )
            )
            /
            k
        )

    return out


def query_metrics(
    score,
    phase,
):
    fdist = future_distance(
        phase[
            "q_future"
        ],
        phase[
            "cand_future"
        ],
    )

    selected = topk_from_scores(
        score,
        TOP_K,
    )

    selected_dist = gather2(
        fdist,
        selected,
    )

    selected_future = gather3(
        phase[
            "cand_future"
        ],
        selected,
    )

    pred = selected_future.mean(
        axis=1
    )

    forecast_mse = np.mean(
        (
            pred -
            phase[
                "q_future"
            ]
        ) ** 2,
        axis=1,
    )

    selected_channel = gather2(
        phase[
            "cand_channel"
        ],
        selected,
    )

    other_frac = np.mean(
        (
            selected_channel
            !=
            phase[
                "q_channel"
            ][
                :,
                None
            ]
        ),
        axis=1,
    )

    return pd.DataFrame({
        "AnalogFutureMSE":
            selected_dist.mean(
                axis=1
            ).astype(
                np.float32
            ),

        "RetrievalForecastMSE":
            forecast_mse.astype(
                np.float32
            ),

        "NDCG@K":
            ndcg_at_k(
                score,
                fdist,
                TOP_K,
            ),

        "OracleRecall@K":
            oracle_recall_at_k(
                selected,
                fdist,
                TOP_K,
            ),

        "OtherChannelFrac@K":
            other_frac.astype(
                np.float32
            ),
    })


## 15. Handcrafted Context baseline

In [ ]:

def context_distance(
    q_context,
    cand_context,
):
    return np.sqrt(
        np.mean(
            (
                cand_context -
                q_context[
                    :,
                    None,
                    :
                ]
            ) ** 2,
            axis=2,
        )
    )


def handcrafted_score(
    phase,
    lam,
):
    return (
        phase[
            "pattern_score"
        ]
        -
        lam *
        context_distance(
            phase[
                "q_context"
            ],
            phase[
                "cand_context"
            ],
        )
    ).astype(
        np.float32
    )


def select_hand_lambda(
    val_phase,
):
    rows = []

    for lam in HAND_LAMBDA_GRID:

        m = query_metrics(
            handcrafted_score(
                val_phase,
                lam,
            ),
            val_phase,
        )

        rows.append({
            "Lambda":
                lam,

            "ValAnalogFutureMSE":
                float(
                    m[
                        "AnalogFutureMSE"
                    ].mean()
                ),

            "ValNDCG@K":
                float(
                    m[
                        "NDCG@K"
                    ].mean()
                ),
        })

    table = pd.DataFrame(
        rows
    )

    best = (
        table
        .sort_values(
            [
                "ValAnalogFutureMSE",
                "ValNDCG@K",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .iloc[
            0
        ]
    )

    return (
        float(
            best[
                "Lambda"
            ]
        ),
        table,
    )


## 16. Future-Compatible reranker

In [ ]:

class FutureCompatibleReranker(
    nn.Module
):
    def __init__(
        self,
        context_dim,
        hidden_dim=128,
        dropout=0.10,
        initial_alpha=0.10,
    ):
        super().__init__()

        input_dim = (
            1 +
            4 *
            context_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                hidden_dim //
                2,
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim //
                2,
                1,
            ),
        )

        raw_alpha = math.log(
            math.exp(
                initial_alpha
            ) -
            1.0
        )

        self.raw_alpha = nn.Parameter(
            torch.tensor(
                raw_alpha,
                dtype=torch.float32,
            )
        )

    @property
    def alpha(
        self
    ):
        return F.softplus(
            self.raw_alpha
        )

    def forward(
        self,
        pattern_score,
        q_context,
        cand_context,
    ):
        B, M, D = (
            cand_context.shape
        )

        q = (
            q_context[
                :,
                None,
                :
            ]
            .expand(
                -1,
                M,
                -1,
            )
        )

        diff = (
            q -
            cand_context
        )

        feat = torch.cat(
            [
                pattern_score[
                    ...,
                    None
                ],
                q,
                cand_context,
                diff,
                diff.abs(),
            ],
            dim=-1,
        )

        delta = (
            self.mlp(
                feat
            )
            .squeeze(
                -1
            )
        )

        return (
            pattern_score +
            self.alpha *
            delta
        )


## 17. Listwise future-supervision loss

In [ ]:

def listwise_future_loss(
    score,
    future_dist,
):
    mean = future_dist.mean(
        dim=1,
        keepdim=True,
    )

    std = future_dist.std(
        dim=1,
        keepdim=True,
        unbiased=False,
    ).clamp_min(
        1e-6
    )

    z = (
        future_dist -
        mean
    ) / std

    target = torch.softmax(
        -z /
        TAU_Y,
        dim=1,
    )

    log_prob = F.log_softmax(
        score,
        dim=1,
    )

    loss = -(
        target *
        log_prob
    ).sum(
        dim=1
    ).mean()

    assert torch.isfinite(
        loss
    )

    return loss


## 18. Training helpers

In [ ]:

def set_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


def shuffled_future_array(
    phase,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    perm = rng.permutation(
        len(
            phase[
                "q_future"
            ]
        )
    )

    return phase[
        "q_future"
    ][
        perm
    ]


def train_one_phase(
    model,
    optimizer,
    phase,
    shuffled_future=None,
):
    model.train()

    n = len(
        phase[
            "q_idx"
        ]
    )

    order = np.random.permutation(
        n
    )

    losses = []

    for start in range(
        0,
        n,
        TRAIN_BATCH,
    ):

        ids = order[
            start:
            start +
            TRAIN_BATCH
        ]

        ps = torch.tensor(
            phase[
                "pattern_score"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qc = torch.tensor(
            phase[
                "q_context"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cc = torch.tensor(
            phase[
                "cand_context"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cf = torch.tensor(
            phase[
                "cand_future"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qf_np = (
            phase[
                "q_future"
            ][
                ids
            ]
            if shuffled_future is None
            else
            shuffled_future[
                ids
            ]
        )

        qf = torch.tensor(
            qf_np,
            dtype=torch.float32,
            device=DEVICE,
        )

        dist = (
            (
                cf -
                qf[
                    :,
                    None,
                    :
                ]
            ) ** 2
        ).mean(
            dim=2
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        score = model(
            ps,
            qc,
            cc,
        )

        loss = listwise_future_loss(
            score,
            dist,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0,
        )

        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def predict_scores(
    model,
    phase,
):
    model.eval()

    out = []

    n = len(
        phase[
            "q_idx"
        ]
    )

    for start in range(
        0,
        n,
        EVAL_BATCH,
    ):

        end = min(
            start +
            EVAL_BATCH,
            n,
        )

        ps = torch.tensor(
            phase[
                "pattern_score"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qc = torch.tensor(
            phase[
                "q_context"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cc = torch.tensor(
            phase[
                "cand_context"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        out.append(
            model(
                ps,
                qc,
                cc,
            ).cpu()
        )

    return (
        torch.cat(
            out,
            dim=0,
        ).numpy().astype(
            np.float32
        )
    )


## 19. Phase A: select epoch on Validation

In [ ]:

def select_epoch(
    dataset_name,
    H,
    seed,
    shuffled=False,
):
    set_seed(
        seed
    )

    train_phase = phase_data(
        dataset_name,
        H,
        "train",
    )

    val_phase = phase_data(
        dataset_name,
        H,
        "val",
    )

    model = FutureCompatibleReranker(
        context_dim=CONTEXT_DIM
    ).to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    shuffled_train = None

    if shuffled:

        shuffled_train = shuffled_future_array(
            train_phase,
            seed=(
                100000 +
                seed
            ),
        )

    best_epoch = None
    best_val = float(
        "inf"
    )

    wait = 0
    rows = []

    for epoch in range(
        1,
        MAX_EPOCHS +
        1,
    ):

        train_loss = train_one_phase(
            model,
            optimizer,
            train_phase,
            shuffled_future=shuffled_train,
        )

        val_score = predict_scores(
            model,
            val_phase,
        )

        val_m = query_metrics(
            val_score,
            val_phase,
        )

        val_analog = float(
            val_m[
                "AnalogFutureMSE"
            ].mean()
        )

        rows.append({
            "Epoch":
                epoch,

            "TrainLoss":
                train_loss,

            "ValAnalogFutureMSE":
                val_analog,

            "ValNDCG@K":
                float(
                    val_m[
                        "NDCG@K"
                    ].mean()
                ),

            "Alpha":
                float(
                    model.alpha.item()
                ),
        })

        if (
            val_analog <
            best_val -
            1e-10
        ):

            best_val = (
                val_analog
            )

            best_epoch = (
                epoch
            )

            wait = 0

        else:

            wait += 1

        if wait >= PATIENCE:
            break

    assert best_epoch is not None

    return (
        best_epoch,
        best_val,
        pd.DataFrame(
            rows
        ),
    )


## 20. Phase B: refit from scratch on Train + Validation supervision

In [ ]:

def refit_model(
    dataset_name,
    H,
    seed,
    epochs,
    shuffled=False,
):
    set_seed(
        seed
    )

    train_phase = phase_data(
        dataset_name,
        H,
        "train",
    )

    val_phase = phase_data(
        dataset_name,
        H,
        "val",
    )

    model = FutureCompatibleReranker(
        context_dim=CONTEXT_DIM
    ).to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    shuffled_train = None
    shuffled_val = None

    if shuffled:

        shuffled_train = shuffled_future_array(
            train_phase,
            seed=(
                200000 +
                seed
            ),
        )

        shuffled_val = shuffled_future_array(
            val_phase,
            seed=(
                300000 +
                seed
            ),
        )

    for epoch in range(
        1,
        epochs +
        1,
    ):

        _ = train_one_phase(
            model,
            optimizer,
            train_phase,
            shuffled_future=shuffled_train,
        )

        _ = train_one_phase(
            model,
            optimizer,
            val_phase,
            shuffled_future=shuffled_val,
        )

    return model


## 21. Candidate-pool oracle

In [ ]:

def oracle_within_m_metrics(
    phase,
):
    fdist = future_distance(
        phase[
            "q_future"
        ],
        phase[
            "cand_future"
        ],
    )

    oracle_idx = np.argpartition(
        fdist,
        kth=TOP_K -
        1,
        axis=1,
    )[
        :,
        :TOP_K
    ]

    oracle_dist = gather2(
        fdist,
        oracle_idx,
    ).mean(
        axis=1
    )

    oracle_future = gather3(
        phase[
            "cand_future"
        ],
        oracle_idx,
    )

    oracle_pred = oracle_future.mean(
        axis=1
    )

    oracle_forecast = np.mean(
        (
            oracle_pred -
            phase[
                "q_future"
            ]
        ) ** 2,
        axis=1,
    )

    n = len(
        oracle_dist
    )

    return pd.DataFrame({
        "AnalogFutureMSE":
            oracle_dist.astype(
                np.float32
            ),

        "RetrievalForecastMSE":
            oracle_forecast.astype(
                np.float32
            ),

        "NDCG@K":
            np.ones(
                n,
                dtype=np.float32,
            ),

        "OracleRecall@K":
            np.ones(
                n,
                dtype=np.float32,
            ),

        "OtherChannelFrac@K":
            np.full(
                n,
                np.nan,
                dtype=np.float32,
            ),
    })


## 22. Run all 12 tasks

In [ ]:

SUMMARY_ROWS = []
QUERY_RESULTS = {}

training_history_frames = []
lambda_frames = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        print(
            "\n" +
            "=" *
            100
        )

        print(
            f"{dataset_name} | H={H}"
        )

        print(
            "=" *
            100
        )

        val_phase = phase_data(
            dataset_name,
            H,
            "val",
        )

        test_phase = phase_data(
            dataset_name,
            H,
            "test",
        )

        # -------------------------------------------------------------
        # Pattern
        # -------------------------------------------------------------

        pattern_metrics = query_metrics(
            test_phase[
                "pattern_score"
            ],
            test_phase,
        )

        # -------------------------------------------------------------
        # Handcrafted context
        # -------------------------------------------------------------

        best_lambda, lambda_table = select_hand_lambda(
            val_phase
        )

        lambda_table[
            "Dataset"
        ] = dataset_name

        lambda_table[
            "Horizon"
        ] = H

        lambda_frames.append(
            lambda_table
        )

        hand_metrics = query_metrics(
            handcrafted_score(
                test_phase,
                best_lambda,
            ),
            test_phase,
        )

        # -------------------------------------------------------------
        # Learned + ShuffledFuture
        # -------------------------------------------------------------

        mean_seed_metrics = {}

        for method_name, shuffled in [
            (
                "Learned",
                False,
            ),
            (
                "ShuffledFuture",
                True,
            ),
        ]:

            seed_frames = []

            for seed in SEEDS:

                model_path = (
                    MODEL_DIR /
                    f"{dataset_name}_H{H}_{method_name}_seed{seed}.pt"
                )

                history_path = (
                    MODEL_DIR /
                    f"{dataset_name}_H{H}_{method_name}_seed{seed}_history.csv"
                )

                if (
                    model_path.exists()
                    and
                    not
                    FORCE_RETRAIN
                ):

                    ckpt = torch.load(
                        model_path,
                        map_location="cpu",
                        weights_only=False,
                    )

                    best_epoch = int(
                        ckpt[
                            "BestEpoch"
                        ]
                    )

                    best_val = float(
                        ckpt[
                            "BestValidationAnalogFutureMSE"
                        ]
                    )

                    model = FutureCompatibleReranker(
                        context_dim=CONTEXT_DIM
                    ).to(
                        DEVICE
                    )

                    model.load_state_dict(
                        ckpt[
                            "StateDict"
                        ]
                    )

                    print(
                        method_name,
                        "seed",
                        seed,
                        "| loaded | epoch",
                        best_epoch,
                        "| val",
                        best_val,
                    )

                else:

                    (
                        best_epoch,
                        best_val,
                        history,
                    ) = select_epoch(
                        dataset_name,
                        H,
                        seed,
                        shuffled=shuffled,
                    )

                    history[
                        "Dataset"
                    ] = dataset_name

                    history[
                        "Horizon"
                    ] = H

                    history[
                        "Method"
                    ] = method_name

                    history[
                        "Seed"
                    ] = seed

                    history.to_csv(
                        history_path,
                        index=False,
                    )

                    training_history_frames.append(
                        history
                    )

                    model = refit_model(
                        dataset_name,
                        H,
                        seed,
                        best_epoch,
                        shuffled=shuffled,
                    )

                    torch.save(
                        {
                            "Dataset":
                                dataset_name,

                            "Horizon":
                                H,

                            "Method":
                                method_name,

                            "Seed":
                                seed,

                            "BestEpoch":
                                best_epoch,

                            "BestValidationAnalogFutureMSE":
                                best_val,

                            "StateDict":
                                model.state_dict(),

                            "TopM":
                                TOP_M,

                            "TopK":
                                TOP_K,

                            "TauY":
                                TAU_Y,

                            "Protocol":
                                (
                                    "Phase A: train-query fit, "
                                    "validation epoch selection; "
                                    "Phase B: refit from scratch using "
                                    "train + validation retrieval supervision."
                                ),
                        },
                        model_path,
                    )

                    print(
                        method_name,
                        "seed",
                        seed,
                        "| trained | epoch",
                        best_epoch,
                        "| val",
                        best_val,
                    )

                score = predict_scores(
                    model,
                    test_phase,
                )

                seed_frames.append(
                    query_metrics(
                        score,
                        test_phase,
                    )
                )

                del model

                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            mean_seed_metrics[
                method_name
            ] = pd.DataFrame({
                col:
                    np.stack(
                        [
                            f[
                                col
                            ].to_numpy()
                            for f
                            in seed_frames
                        ],
                        axis=0,
                    ).mean(
                        axis=0
                    )

                for col in seed_frames[
                    0
                ].columns
            })

        oracle_metrics = oracle_within_m_metrics(
            test_phase
        )

        method_metrics = {
            "Pattern":
                pattern_metrics,

            "Handcrafted":
                hand_metrics,

            "Learned":
                mean_seed_metrics[
                    "Learned"
                ],

            "ShuffledFuture":
                mean_seed_metrics[
                    "ShuffledFuture"
                ],

            "OracleWithinM":
                oracle_metrics,
        }

        # -------------------------------------------------------------
        # Query-level paired table
        # -------------------------------------------------------------

        q = pd.DataFrame({
            "Anchor":
                test_phase[
                    "q_anchor"
                ],

            "ChannelIndex":
                test_phase[
                    "q_channel"
                ],

            "ShiftScore":
                np.mean(
                    np.abs(
                        test_phase[
                            "q_context"
                        ]
                    ),
                    axis=1,
                ),
        })

        for method_name, m in method_metrics.items():

            for col in m.columns:

                q[
                    f"{method_name}_{col}"
                ] = m[
                    col
                ].to_numpy()

        QUERY_RESULTS[
            (
                dataset_name,
                H
            )
        ] = q

        q.to_parquet(
            RESULT_DIR /
            f"query_level_{dataset_name}_H{H}.parquet",
            index=False,
        )

        # -------------------------------------------------------------
        # Main task summary
        # -------------------------------------------------------------

        for method_name, m in method_metrics.items():

            SUMMARY_ROWS.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Method":
                    method_name,

                "HandLambda":
                    (
                        best_lambda
                        if method_name ==
                        "Handcrafted"
                        else
                        np.nan
                    ),

                "AnalogFutureMSE":
                    float(
                        m[
                            "AnalogFutureMSE"
                        ].mean()
                    ),

                "RetrievalForecastMSE":
                    float(
                        m[
                            "RetrievalForecastMSE"
                        ].mean()
                    ),

                "NDCG@K":
                    float(
                        m[
                            "NDCG@K"
                        ].mean()
                    ),

                "OracleRecall@K":
                    float(
                        m[
                            "OracleRecall@K"
                        ].mean()
                    ),

                "OtherChannelFrac@K":
                    float(
                        np.nanmean(
                            m[
                                "OtherChannelFrac@K"
                            ]
                        )
                    ),
            })


summary_table = pd.DataFrame(
    SUMMARY_ROWS
)

display(
    summary_table.sort_values(
        [
            "Dataset",
            "Horizon",
            "AnalogFutureMSE",
        ]
    )
)

summary_table.to_csv(
    RESULT_DIR /
    "03_main_cross_domain_summary.csv",
    index=False,
)

if training_history_frames:

    pd.concat(
        training_history_frames,
        ignore_index=True,
    ).to_csv(
        RESULT_DIR /
        "04_training_history.csv",
        index=False,
    )

pd.concat(
    lambda_frames,
    ignore_index=True,
).to_csv(
    RESULT_DIR /
    "05_handcrafted_lambda_selection.csv",
    index=False,
)


## 23. Relative improvements

In [ ]:

improvement_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        t = (
            summary_table[
                (
                    summary_table[
                        "Dataset"
                    ] ==
                    dataset_name
                )
                &
                (
                    summary_table[
                        "Horizon"
                    ] ==
                    H
                )
            ]
            .set_index(
                "Method"
            )
        )

        learned = t.loc[
            "Learned"
        ]

        for baseline in [
            "Pattern",
            "Handcrafted",
            "ShuffledFuture",
        ]:

            b = t.loc[
                baseline
            ]

            improvement_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Baseline":
                    baseline,

                "AnalogFutureMSE_Improvement_%":
                    100.0 *
                    (
                        b[
                            "AnalogFutureMSE"
                        ] -
                        learned[
                            "AnalogFutureMSE"
                        ]
                    ) /
                    b[
                        "AnalogFutureMSE"
                    ],

                "RetrievalForecastMSE_Improvement_%":
                    100.0 *
                    (
                        b[
                            "RetrievalForecastMSE"
                        ] -
                        learned[
                            "RetrievalForecastMSE"
                        ]
                    ) /
                    b[
                        "RetrievalForecastMSE"
                    ],

                "NDCG_AbsoluteGain":
                    (
                        learned[
                            "NDCG@K"
                        ] -
                        b[
                            "NDCG@K"
                        ]
                    ),

                "OracleRecall_AbsoluteGain":
                    (
                        learned[
                            "OracleRecall@K"
                        ] -
                        b[
                            "OracleRecall@K"
                        ]
                    ),
            })

improvement_table = pd.DataFrame(
    improvement_rows
)

display(
    improvement_table
)

improvement_table.to_csv(
    RESULT_DIR /
    "06_relative_improvements.csv",
    index=False,
)


## 24. Shift-quartile analysis

In [ ]:

shift_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        q = QUERY_RESULTS[
            (
                dataset_name,
                H
            )
        ].copy()

        q[
            "ShiftQuartile"
        ] = pd.qcut(
            q[
                "ShiftScore"
            ],
            q=4,
            labels=[
                "Q1_low",
                "Q2",
                "Q3",
                "Q4_high",
            ],
            duplicates="drop",
        )

        for quartile, g in q.groupby(
            "ShiftQuartile",
            observed=True,
        ):

            pattern = float(
                g[
                    "Pattern_AnalogFutureMSE"
                ].mean()
            )

            learned = float(
                g[
                    "Learned_AnalogFutureMSE"
                ].mean()
            )

            hand = float(
                g[
                    "Handcrafted_AnalogFutureMSE"
                ].mean()
            )

            shuffled = float(
                g[
                    "ShuffledFuture_AnalogFutureMSE"
                ].mean()
            )

            shift_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "ShiftQuartile":
                    str(
                        quartile
                    ),

                "N":
                    len(
                        g
                    ),

                "MeanShiftScore":
                    float(
                        g[
                            "ShiftScore"
                        ].mean()
                    ),

                "Learned_vs_Pattern_Improvement_%":
                    100.0 *
                    (
                        pattern -
                        learned
                    ) /
                    pattern,

                "Learned_vs_Handcrafted_Improvement_%":
                    100.0 *
                    (
                        hand -
                        learned
                    ) /
                    hand,

                "Learned_vs_Shuffled_Improvement_%":
                    100.0 *
                    (
                        shuffled -
                        learned
                    ) /
                    shuffled,
            })

shift_table = pd.DataFrame(
    shift_rows
)

display(
    shift_table
)

shift_table.to_csv(
    RESULT_DIR /
    "07_shift_quartile_analysis.csv",
    index=False,
)


## 25. Moving-block bootstrap

In [ ]:

def moving_block_bootstrap(
    x,
    block_len,
    n_boot,
    seed,
):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    n = len(
        x
    )

    if n < block_len:

        return {
            "ObservedImprovement":
                float(
                    x.mean()
                ),

            "CI_2.5%":
                np.nan,

            "CI_97.5%":
                np.nan,

            "P_gt_0":
                np.nan,

            "NAnchors":
                n,
        }

    rng = np.random.default_rng(
        seed
    )

    n_blocks = math.ceil(
        n /
        block_len
    )

    max_start = (
        n -
        block_len
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):

        parts = []

        for _ in range(
            n_blocks
        ):

            s = rng.integers(
                0,
                max_start +
                1,
            )

            parts.append(
                x[
                    s:
                    s +
                    block_len
                ]
            )

        sample = np.concatenate(
            parts
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "ObservedImprovement":
            float(
                x.mean()
            ),

        "CI_2.5%":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),

        "CI_97.5%":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),

        "P_gt_0":
            float(
                (
                    boot >
                    0
                ).mean()
            ),

        "NAnchors":
            n,
    }


bootstrap_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        q = QUERY_RESULTS[
            (
                dataset_name,
                H
            )
        ]

        for baseline in [
            "Pattern",
            "Handcrafted",
            "ShuffledFuture",
        ]:

            for metric in [
                "AnalogFutureMSE",
                "RetrievalForecastMSE",
            ]:

                tmp = pd.DataFrame({
                    "Anchor":
                        q[
                            "Anchor"
                        ],

                    "Diff":
                        (
                            q[
                                f"{baseline}_{metric}"
                            ]
                            -
                            q[
                                f"Learned_{metric}"
                            ]
                        ),
                })

                anchor_diff = (
                    tmp
                    .groupby(
                        "Anchor"
                    )[
                        "Diff"
                    ]
                    .mean()
                    .sort_index()
                    .to_numpy()
                )

                seed = (
                    DATASET_SEED[
                        dataset_name
                    ] +
                    H *
                    100 +
                    len(
                        bootstrap_rows
                    )
                )

                r = moving_block_bootstrap(
                    anchor_diff,
                    block_len=BLOCK_ANCHORS,
                    n_boot=N_BOOT,
                    seed=seed,
                )

                r.update({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "Baseline":
                        baseline,

                    "Proposed":
                        "Learned",

                    "Metric":
                        metric,

                    "SignificantImprovement":
                        bool(
                            (
                                not
                                np.isnan(
                                    r[
                                        "CI_2.5%"
                                    ]
                                )
                            )
                            and
                            (
                                r[
                                    "CI_2.5%"
                                ] >
                                0
                            )
                        ),
                })

                bootstrap_rows.append(
                    r
                )

bootstrap_table = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_table
)

bootstrap_table.to_csv(
    RESULT_DIR /
    "08_moving_block_bootstrap.csv",
    index=False,
)


## 26. Cross-domain win counts

In [ ]:

win_rows = []

for baseline in [
    "Pattern",
    "Handcrafted",
    "ShuffledFuture",
]:

    sub = improvement_table[
        improvement_table[
            "Baseline"
        ] ==
        baseline
    ]

    b_analog = bootstrap_table[
        (
            bootstrap_table[
                "Baseline"
            ] ==
            baseline
        )
        &
        (
            bootstrap_table[
                "Metric"
            ] ==
            "AnalogFutureMSE"
        )
    ]

    b_forecast = bootstrap_table[
        (
            bootstrap_table[
                "Baseline"
            ] ==
            baseline
        )
        &
        (
            bootstrap_table[
                "Metric"
            ] ==
            "RetrievalForecastMSE"
        )
    ]

    win_rows.append({
        "Baseline":
            baseline,

        "Tasks":
            len(
                sub
            ),

        "AnalogBetterTasks":
            int(
                (
                    sub[
                        "AnalogFutureMSE_Improvement_%"
                    ] >
                    0
                ).sum()
            ),

        "AnalogSignificantTasks":
            int(
                b_analog[
                    "SignificantImprovement"
                ].sum()
            ),

        "ForecastBetterTasks":
            int(
                (
                    sub[
                        "RetrievalForecastMSE_Improvement_%"
                    ] >
                    0
                ).sum()
            ),

        "ForecastSignificantTasks":
            int(
                b_forecast[
                    "SignificantImprovement"
                ].sum()
            ),

        "NDCGBetterTasks":
            int(
                (
                    sub[
                        "NDCG_AbsoluteGain"
                    ] >
                    0
                ).sum()
            ),

        "OracleRecallBetterTasks":
            int(
                (
                    sub[
                        "OracleRecall_AbsoluteGain"
                    ] >
                    0
                ).sum()
            ),

        "MeanAnalogImprovement_%":
            float(
                sub[
                    "AnalogFutureMSE_Improvement_%"
                ].mean()
            ),

        "MedianAnalogImprovement_%":
            float(
                sub[
                    "AnalogFutureMSE_Improvement_%"
                ].median()
            ),
    })

win_table = pd.DataFrame(
    win_rows
)

display(
    win_table
)

win_table.to_csv(
    RESULT_DIR /
    "09_cross_domain_win_counts.csv",
    index=False,
)


## 27. High-shift vs low-shift effect

In [ ]:

shift_effect_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        t = shift_table[
            (
                shift_table[
                    "Dataset"
                ] ==
                dataset_name
            )
            &
            (
                shift_table[
                    "Horizon"
                ] ==
                H
            )
        ].set_index(
            "ShiftQuartile"
        )

        if (
            "Q1_low"
            not in t.index
            or
            "Q4_high"
            not in t.index
        ):
            continue

        low = float(
            t.loc[
                "Q1_low",
                "Learned_vs_Pattern_Improvement_%",
            ]
        )

        high = float(
            t.loc[
                "Q4_high",
                "Learned_vs_Pattern_Improvement_%",
            ]
        )

        shift_effect_rows.append({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            "LowShiftImprovement_%":
                low,

            "HighShiftImprovement_%":
                high,

            "HighMinusLow_pp":
                high -
                low,

            "HigherBenefitUnderShift":
                bool(
                    high >
                    low
                ),
        })

shift_effect_table = pd.DataFrame(
    shift_effect_rows
)

display(
    shift_effect_table
)

shift_effect_table.to_csv(
    RESULT_DIR /
    "10_shift_effect_summary.csv",
    index=False,
)


## 28. Final decision

In [ ]:

pattern = improvement_table[
    improvement_table[
        "Baseline"
    ] ==
    "Pattern"
]

shuffled = improvement_table[
    improvement_table[
        "Baseline"
    ] ==
    "ShuffledFuture"
]

pattern_boot = bootstrap_table[
    (
        bootstrap_table[
            "Baseline"
        ] ==
        "Pattern"
    )
    &
    (
        bootstrap_table[
            "Metric"
        ] ==
        "AnalogFutureMSE"
    )
]

n_tasks = len(
    pattern
)

pattern_wins = int(
    (
        pattern[
            "AnalogFutureMSE_Improvement_%"
        ] >
        0
    ).sum()
)

pattern_sig = int(
    pattern_boot[
        "SignificantImprovement"
    ].sum()
)

shuffle_wins = int(
    (
        shuffled[
            "AnalogFutureMSE_Improvement_%"
        ] >
        0
    ).sum()
)

forecast_wins = int(
    (
        pattern[
            "RetrievalForecastMSE_Improvement_%"
        ] >
        0
    ).sum()
)

ndcg_wins = int(
    (
        pattern[
            "NDCG_AbsoluteGain"
        ] >
        0
    ).sum()
)

recall_wins = int(
    (
        pattern[
            "OracleRecall_AbsoluteGain"
        ] >
        0
    ).sum()
)

shift_wins = int(
    shift_effect_table[
        "HigherBenefitUnderShift"
    ].sum()
)

if (
    pattern_wins >=
    9
    and
    pattern_sig >=
    6
    and
    shuffle_wins >=
    9
    and
    ndcg_wins >=
    9
):

    recommendation = (
        "Strong cross-domain evidence. "
        "Proceed to 5-seed final replication, "
        "theoretical proposition, and ICLR manuscript."
    )

elif pattern_wins >= 7:

    recommendation = (
        "Promising but heterogeneous evidence. "
        "Analyze failure/success regimes before finalizing framing."
    )

else:

    recommendation = (
        "Cross-domain generalization is too weak for the broad ICLR claim. "
        "Reconsider scope rather than increasing model complexity."
    )


decision_table = pd.DataFrame(
    [
        {
            "TotalTasks":
                n_tasks,

            "LearnedBeatsPattern_Analog":
                pattern_wins,

            "SignificantPatternWins_Analog":
                pattern_sig,

            "LearnedBeatsShuffled_Analog":
                shuffle_wins,

            "LearnedBeatsPattern_Forecast":
                forecast_wins,

            "LearnedBeatsPattern_NDCG":
                ndcg_wins,

            "LearnedBeatsPattern_OracleRecall":
                recall_wins,

            "HigherBenefitUnderShiftTasks":
                shift_wins,

            "MeanPatternAnalogImprovement_%":
                float(
                    pattern[
                        "AnalogFutureMSE_Improvement_%"
                    ].mean()
                ),

            "MedianPatternAnalogImprovement_%":
                float(
                    pattern[
                        "AnalogFutureMSE_Improvement_%"
                    ].median()
                ),

            "Recommendation":
                recommendation,
        }
    ]
)

display(
    decision_table
)

decision_table.to_csv(
    RESULT_DIR /
    "11_final_decision_summary.csv",
    index=False,
)


# Output guide

Generated artifacts are written under

```text
_work/cross_domain_clean/
```

The most useful summary files are the cross-domain method comparison, relative improvements, moving-block bootstrap results, and win-count summaries produced near the end of the notebook.

For the public repository workflow, continue with:

```text
00_cross_domain_base.ipynb
    -> 01_etth1_weather_relevance.ipynb
    -> 02_candidate_prior.ipynb
```

The final confirmatory pipeline is independent and begins with `experiments/confirmatory/confirmatory_benchmark.ipynb`.
